In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    to_date,
    when
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_restructuring_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Restructuring"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = ["PortfolioName", "Prefix", "unit_code_profits"]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )


        # ================== eaisig_mod ==================

        eaisig_mod_extra_filters = None

        if eaisig_mod_extra_filters is not None:
            eaisig_mod_extra_filters.append(date_filter)
            eaisig_mod_final_filters = " AND ".join(eaisig_mod_extra_filters)
        else:
            eaisig_mod_final_filters = date_filter

        eaisig_mod_selected_fields = [
            "eaisig_mod_case_s_id",
            "eaisig_mod_old_account_id",
            "eaisig_mod_link_type",
            "eaisig_mod_case_contrnum_old",
            "eaisig_mod_case_contrnum_new",
            "eaisig_mod_cust_s_unid",
            "eaisig_mod_settlement_prd_id",
            "eaisig_mod_settlement_date",
            "eaisig_mod_restr_reason_code",
            "eaisig_mod_settlement_amount",
            "eaisig_mod_nonviable_amount",
            "fullrow",
            "eaisig_mod_createdt",
            "businessdate",
        ]

        eaisig_mod = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_eaisig_mod",
            eaisig_mod_final_filters,
            eaisig_mod_selected_fields,
        )

        # =============== Join with portfolio ===============
        eaisig_mod = eaisig_mod.join(
            portfolio,
            on=substring(eaisig_mod["eaisig_mod_cust_s_unid"], 1, 3) == portfolio["Prefix"],
            how="left"
        )

        # HashKeys
        eaisig_mod_primary_key_fields = [
            eaisig_mod.eaisig_mod_case_s_id,
            eaisig_mod.eaisig_mod_old_account_id,
            eaisig_mod.eaisig_mod_link_type,
            lit("arotron"),
        ]

        eaisig_mod = eaisig_mod.withColumn(
            "RestructuringId",
            sha2(concat_ws("", *eaisig_mod_primary_key_fields), 256),
        ).withColumn(
            "RestructuringKey",
            concat_ws(
                ";",
                eaisig_mod.eaisig_mod_case_s_id,
                eaisig_mod.eaisig_mod_old_account_id,
                eaisig_mod.eaisig_mod_link_type)
        ).withColumn(
            "RestructuringDate",
            to_date(
                when(eaisig_mod.eaisig_mod_settlement_date == "", lit(None))
                .otherwise(eaisig_mod.eaisig_mod_settlement_date)
            )
        )

        # ================== Selection of Final Columns ==================

        result_df_arotron = eaisig_mod.select(
            eaisig_mod.RestructuringId.cast("string"),
            lit("arotron").alias("SourceSystem"),
            coalesce(col("portfolio.unit_code_profits"), lit(-9998)).alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Wholesale")).cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            eaisig_mod.RestructuringKey.cast("string"),
            eaisig_mod.eaisig_mod_case_contrnum_old.cast("string").alias(
                "OldAgreementKey"
            ),
            eaisig_mod.eaisig_mod_case_contrnum_new.cast("string").alias(
                "NewAgreementKey"
            ),
            eaisig_mod.eaisig_mod_old_account_id.cast("string").alias("OldExposureKey"),
            eaisig_mod.eaisig_mod_case_s_id.cast("string").alias("NewExposureKey"),
            lit(None).alias("WriteOffExposureKey"),
            lit(None).alias("InterestExposureKey"),
            lit(None).alias("InterestFreeExposureKey"),
            eaisig_mod.eaisig_mod_cust_s_unid.cast("string").alias("PartyKey"),
            eaisig_mod.eaisig_mod_cust_s_unid.cast("string").alias("CDI"),
            eaisig_mod.eaisig_mod_settlement_prd_id.cast("string").alias(
                "RestructuringProductKey"
            ),
            eaisig_mod.eaisig_mod_link_type.cast("string").alias(
                "RestructuringLinkType"
            ),
            lit(None).alias("CompletionTargetDate"),
            eaisig_mod.RestructuringDate.alias("RestructuringDate"),
            eaisig_mod.eaisig_mod_restr_reason_code.cast("string").alias(
                "RestructuringTypeCode"
            ),
            eaisig_mod.eaisig_mod_restr_reason_code.cast("string").alias(
                "RestructuringType"
            ),
            eaisig_mod.eaisig_mod_settlement_amount.cast("decimal(19,2)").alias(
                "RestructuringAmount"
            ),
            lit(None).alias("WriteOffAmount"),
            eaisig_mod.eaisig_mod_nonviable_amount.cast("decimal(19,2)").alias(
                "RestructuringDiscountAmount"
            ),
            lit(None).alias("SpreadPercentage"),
            eaisig_mod.fullrow.cast("string").alias("Comment"),
            eaisig_mod.eaisig_mod_createdt.cast("timestamp").alias("CreatedDate"),
            lit(None).alias("CreatedBy"),
            lit(None).alias("LastUpdatedDate"),
            lit(None).alias("LastUpdatedBy"),
            eaisig_mod.businessdate.alias("BusinessDate"),
        )

        result_df_arotron.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_restructuring_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)